# Project 1 - Part 1: Classification

This notebook contains the data analysis and model building process using machine learning classification algorithms.

## Project Goals:
- Analyze and visualize the dataset
- Apply different classification algorithms
- Compare model performances
- Select and evaluate the best model

In [ ]:
# Loading required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.datasets import load_iris  # For sample dataset
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.style.use('ggplot')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Loading and Exploring the Dataset

In [ ]:
# Load dataset (Using Iris dataset as example)
# You can load your own dataset here:
# data = pd.read_csv('dataset.csv')

# Sample dataset (Iris)
iris = load_iris()
data = pd.DataFrame(iris.data, columns=iris.feature_names)
data['target'] = iris.target
data['target_names'] = data['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

print("Dataset shape:", data.shape)
print("\nFirst 5 rows:")
data.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Basic data analysis
print("General information about the dataset:")
print(data.info())
print("\nMissing value check:")
print(data.isnull().sum())
print("\nClass distribution:")
print(data['target_names'].value_counts())
print("\nStatistical summary:")
data.describe()

In [ ]:
# Data visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Class distribution
data['target_names'].value_counts().plot(kind='bar', ax=axes[0,0])
axes[0,0].set_title('Class Distribution')
axes[0,0].set_xlabel('Classes')
axes[0,0].set_ylabel('Frequency')

# Correlation between features
numeric_data = data.select_dtypes(include=[np.number])
corr_matrix = numeric_data.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', ax=axes[0,1])
axes[0,1].set_title('Feature Correlation')

# Pair plot (for selected features)
feature_cols = data.columns[:-2]  # Exclude last two columns
if len(feature_cols) >= 2:
    data.plot.scatter(x=feature_cols[0], y=feature_cols[1], 
                     c='target', colormap='viridis', ax=axes[1,0])
    axes[1,0].set_title(f'{feature_cols[0]} vs {feature_cols[1]}')

# Box plot
if len(feature_cols) >= 1:
    data.boxplot(column=feature_cols[0], by='target_names', ax=axes[1,1])
    axes[1,1].set_title(f'{feature_cols[0]} Box Plot')
    axes[1,1].set_xlabel('Classes')

plt.tight_layout()
plt.show()

## 3. Data Preprocessing

In [ ]:
# Separating features and target variables
feature_columns = data.columns[:-2]  # Exclude last two columns (target and target_names)
X = data[feature_columns]
y = data['target']

print("Feature columns:", list(X.columns))
print("Target variable unique values:", np.unique(y))

# Splitting data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

# Data standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nData standardization completed.")

## 4. Model Training and Evaluation

In [ ]:
# Defining different classification algorithms
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42),
    'Naive Bayes': GaussianNB(),
    'K-NN': KNeighborsClassifier(n_neighbors=5)
}

# Dictionary to store model results
results = {}

# Training and evaluation for each model
for name, model in models.items():
    print(f"\nTraining {name} model...")
    
    # Model training
    if name == 'SVM' or name == 'K-NN':  # Use standardized data
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:  # Use original data
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    # Accuracy calculation
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    
    print(f"{name} Accuracy: {accuracy:.4f}")
    print(f"\n{name} Detailed Report:")
    print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# Visualizing model performances
plt.figure(figsize=(12, 6))

# Accuracy scores plot
plt.subplot(1, 2, 1)
model_names = list(results.keys())
accuracies = list(results.values())

bars = plt.bar(model_names, accuracies, color=['skyblue', 'lightcoral', 'lightgreen', 'gold'])
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy Score')
plt.ylim(0, 1)

# Write values on top of bars
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{acc:.3f}', ha='center', va='bottom')

plt.xticks(rotation=45)

# Best model's confusion matrix
best_model_name = max(results, key=results.get)
print(f"\nBest model: {best_model_name} (Accuracy: {results[best_model_name]:.4f})")

# Confusion matrix with best model
best_model = models[best_model_name]
if best_model_name == 'SVM' or best_model_name == 'K-NN':
    y_pred_best = best_model.predict(X_test_scaled)
else:
    y_pred_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)
plt.subplot(1, 2, 2)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.title(f'{best_model_name} - Confusion Matrix')
plt.ylabel('True Class')
plt.xlabel('Predicted Class')

plt.tight_layout()
plt.show()

## 5. Results and Evaluation

### Model Performance Summary:
- In this project, we compared different classification algorithms
- The best performing model is shown in the graph above
- Detailed analysis of model predictions was performed with confusion matrix

### Recommendations:
1. Model performance can be improved by collecting more data
2. Hyperparameter optimization can be performed
3. New features can be derived through feature engineering
4. More reliable evaluation can be done with cross-validation

## 6. Next Steps

- [ ] Load your own dataset
- [ ] Adjust data preprocessing steps according to your dataset
- [ ] Perform hyperparameter optimization
- [ ] Apply cross-validation
- [ ] Apply feature selection/feature engineering
- [ ] Save model results